In [1]:
!pip install tensorflow transformers tf-keras
!pip3 install newspaper3k
!pip install lxml[html_clean]
!pip install ipywidgets --upgrade
!pip install tweepy
!pip install requests
!pip install requests_oauthlib


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
from datetime import datetime, timedelta
from newspaper import Article
import nltk

nltk.download('punkt_tab')

# Calculate the start and end dates for this week
end_date = datetime.now()

# Calculate the date for the previous Friday
friday_date = end_date - timedelta(days=end_date.weekday() + 3)  # 3 days before Sunday (weekday = 6)

# Calculate the date for the upcoming Monday
monday_date = end_date - timedelta(days=end_date.weekday())  # This gives you the most recent Monday

print(f"Friday: {friday_date}")
print(f"Monday: {monday_date}")

base_url = "https://api.gdeltproject.org/api/v2/doc/doc"
params = {
    "query": "domain:motorsport.com (Formula One OR F1)",
    "mode": "ArtList",
    "maxrecords": 250,
    "startdatetime": friday_date.strftime("%Y%m%d000000"),  # Format: YYYYMMDDHHMMSS
    "enddatetime": monday_date.strftime("%Y%m%d235959"),      # Format: YYYYMMDDHHMMSS
    "format": "json"
}
'''
    Using gdelt to get all motorsport articles posted this week
'''
response = requests.get(base_url, params=params)

# Check if the request was successful
if response.status_code == 200:
    print("Status Code:", response.status_code)
    articles = response.json()
    # print(articles) 
else:
    print('Articles is empty...')
    articles = {}
'''
    Now all motorsport articles are collected. filter to get all F1 related articles.
'''
f1Articles = []
if 'articles' in articles:
    for article in articles['articles']:
        url = article.get('url', '')
        if "motorsport.com/f1/news" in url:
            f1Articles.append(article)
# print(f1Articles)
'''
    Use Newspaper3k to download the content/text of each article via the url
'''
f1_articles_summaries = []
for article in f1Articles:
    url = article.get('url', '')
    try:
        currentArticle = Article(url)
        currentArticle.download()
        currentArticle.parse()
        f1_articles_summaries.append(currentArticle.text)
    except:
        print("Issue handling article")
# print(f1_articles_summaries)
'''
    Use facebook/bart-large-cnn to summarize EACH article
'''
from transformers import BartForConditionalGeneration, BartTokenizer

model_name = "facebook/bart-large-cnn"
model = BartForConditionalGeneration.from_pretrained(model_name)
tokenizer = BartTokenizer.from_pretrained(model_name)

def summarize_to_bullets(text):
    inputs = tokenizer(text, max_length=1024, return_tensors="pt", truncation=True)
    summary_ids = model.generate(inputs["input_ids"], 
                                 max_length=175, 
                                 min_length=50, 
                                 length_penalty=2.0, 
                                 num_beams=4,
                                 no_repeat_ngram_size=6,
                                 early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

summaryResult = []

for text in f1_articles_summaries:
    summary = summarize_to_bullets(text)
    summaryResult.append(summary)
print('summaryResult:' + str(summaryResult))
print('------------------------------')

'''
    Filter out F1 articles that dont have keywords
'''
filtered = []
filteredText = ''

def filter_relevant_summary(summary, keywords):
    # Check if any keyword is in the summary
    for keyword in keywords:
        if keyword.lower() in summary.lower():
            return True
    return False  # Return None if no relevant keyword found
teams = ['Sauber', 'Alpine', 'Aston Martin', 'Aston', 'Ferrari', 'Haas', 'McLaren', 'Mercedes', 'RedBull', 'Red Bull', 'Williams',
         'Visa Cash App RB', 'VCARB','audi']

drivers = ['bottas', 'zhou', 'guanyu', 'esteban', 'ocon', 'pierre', 'gasly', 'fernando', 'alonso', 'lance',
           'stroll', 'charles', 'leclerc', 'carlos', 'sainz', 'kevin', 'magnussen', 'nico', 'hulkenberg', 'lando', 'norris', 'oscar', 'piastri',
           'lewis', 'hamilton', 'george', 'russell', 'max', 'verstappen', 'sergio', 'perez', 'alex', 'albon', 'logan', 'sargeant', 'yuki', 'tsunoda',
           'nyck', 'de vries', 'franco', 'colapinto','Liam','Lawson']

event = ['pit stops', 'qualifying', 'quali', 'practice sessions', 'driver\'s briefing', 'press conference', 'race strategy',
         'safety car', 'red flag', 'yellow flag', 'track condition', 'penalties', 'penalty', 'tyre choice', 'tire choice','steward',
         'race weekend', 'weather forecasts', 'race result', 'championship standings', 'incidents', 'overtake', 'passes', 'pit lane', 'lap times',
         'reprimand', 'adjustment of', 'tech','disqualified','pit']

keywords = teams + drivers + event

for summary in summaryResult:
    chunk = summary.split('.')
    for x in chunk:
        if filter_relevant_summary(x,keywords):
            filtered.append(x)
            filteredText += x + '. '
print('filtered: ' + filteredText)

'''
    Shorten/summarize the total text to fit twitter's char limit
'''
#Shorten to 280

def shorten(text):
    inputs = tokenizer(text, max_length=1024, return_tensors="pt", truncation=True)
    summary_ids = model.generate(inputs["input_ids"], 
                                 max_length=55,           # max tokens
                                 min_length=20,           # min tokens
                                 length_penalty=2.0,     # > 1 favors shorter summaries
                                 no_repeat_ngram_size=5,  # Avoid repetition and irrelevant sentences
                                 num_beams=5,             # > 4 improves quality and coherence
                                 early_stopping=True
                                )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary
    
if len(filteredText) > 275:
    short = shorten(filteredText)
else:
    short = filteredText
print(short)

'''
    Reformat the shorten summary to be in bullet notation.
'''
# Reformat text
# Split text by period to separate sentences
sentences = short.split('.')

# Format each sentence with a bullet point
bullet_points = "\n".join([f"• {sentence.strip()}" for sentence in sentences if sentence.strip()])

print(bullet_points)
print(len(bullet_points))

'''
    Post to Twitter
'''
import requests
import json
from requests_oauthlib import OAuth1

API_KEY = 'MRcrFnNBKj5U7YKEa7Msm8Oc1'
API_SECRET_KEY = 'YMOWRJCcaWfM9CBMv2jZnFi25LJPHdUt2Eb91qesU0Xa7D1eL4'
ACCESS_TOKEN = '1855362853575782401-YDrWgNT6bDomcXfpeIOIp17JLl0R3C'
ACCESS_TOKEN_SECRET = 'afZb6osNib8DxQ7PBC8Ox3ZeZec8L6pXAQAvzynIupMAH'

# Replace with your Bearer Token
# BEARER_TOKEN = 'AAAAAAAAAAAAAAAAAAAAAESFwwEAAAAAiGAFXA0JvLG6LoH6V8V%2B26bwK3E%3D3NngaDoCGWSNYpgbjZTPQPks2fLEWdX7Tam6N7tafRkamjmfYp'

auth = OAuth1(API_KEY, API_SECRET_KEY, ACCESS_TOKEN, ACCESS_TOKEN_SECRET)

headers = {
  'content-type': 'application/json',
};

# Create the tweet content
tweet_content = {
    "text": f'{bullet_points}'
}

# API URL for posting a tweet
url = 'https://api.x.com/2/tweets'

# Make the POST request to the Twitter API
response = requests.post(url, auth=auth, json=tweet_content, headers=headers)

# Check if the tweet was posted successfully
if response.status_code == 201:
    print("Tweet posted successfully!")
else:
    print(f"Error: {response.status_code} - {response.text}")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\laure\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Friday: 2024-11-08 13:48:55.057963
Monday: 2024-11-11 13:48:55.057963
Status Code: 200
summaryResult:['Audi is on the verge of selling a shareholding in its Formula 1 Sauber team to Qatar. An announcement could come ahead of the Qatar Grand Prix next week. There is also said to be the possibility of an investment that is much larger.', 'Liam Lawson was the only driver with more than 10 grand prix starts to start the race in Brazil. Oliver Bearman was penalised for a 360° pirouette in Turn 7. Franco Colapinto crashed out of the race in safety car conditions as the rain got worse.', 'Aston Martin left the Brazilian Grand Prix scratching its head over problems in the wet race and still short of a solution for its drop of form this year. In the dry last weekend its race pace was the slowest out of everyone. The conclusion is they were triggered by imbalance issues as the result of changing floor specs from qualifying into the race.', 'Aston Martin and Red Bull both have vastly differing re